In [67]:
import numpy as np
import torch
import plotly.graph_objects as go

from src.embedding import embed
from src.gp_ccm import GP_ccm_sig_predict

torch.set_printoptions(sci_mode = False)
# Reproducibility
import watermark

In [68]:
# Python package versions used
%load_ext watermark
%watermark --python
%watermark --iversions

Python implementation: CPython
Python version       : 3.11.3
IPython version      : 8.18.0

watermark : 2.4.3
numpy     : 1.24.1
pandas    : 2.1.3
plotly    : 5.9.0
torch     : 2.1.1+cu118
matplotlib: 3.7.2



In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

# Load data

In [ ]:
co2_norm = torch.load("data/CO2_vostok_stan_400kyr_timeseries.pt").to(torch.float32)
temp_norm = torch.load("data/TEMP_vostok_stan_400kyr_timeseries.pt").to(torch.float32)

# Temp -> CO2

In [ ]:
k = 3
N_TRAIN = torch.tensor([200]).to(device)

##############
### sigCCM ###
##############

sig_filter = torch.ones(size = (k, )).to(device)
sig_shift = torch.tensor(sig_filter.shape[0] - 1).to(device) + 3

NOISE_SCALE = torch.tensor([0.05], device = device)
RBF_SCALE = torch.tensor([0.4], device = device)

y_embeddings, x_gt = embed(
    filter = sig_filter, 
    y = co2_norm, 
    x = temp_norm, 
    max_pos_offset = sig_shift, 
    device = device)

rho, nlml, x_test_mean_est, x_test_covar_est = GP_ccm_sig_predict(
    y_embeddings_train = y_embeddings[0 : N_TRAIN].unsqueeze(-1), # [N, E, 1]
    y_embeddings_test = y_embeddings[N_TRAIN : ].unsqueeze(-1), 
    x_train = x_gt[0 : N_TRAIN], # [N]
    x_test = x_gt[N_TRAIN : ], 
    noise = NOISE_SCALE, 
    rbf_sigma = RBF_SCALE, 
    device = device)

print(rho)

In [ ]:
sym = torch.triu(x_test_covar_est, diagonal = 1).add(torch.triu(x_test_covar_est, diagonal = 1).mT)
# x_test_covar_est_psd = sym.add(torch.eye(n = sym.shape[0], device = device).mul(torch.clip(x_test_covar_est.diag(), min = 0.00001)))
x_test_covar_est_psd = sym.add(torch.eye(n = sym.shape[0], device = device).mul(torch.clip(x_test_covar_est.diag(), min = 0.000).add(0.3)))
# x_test_covar_est_psd_alternative = x_test_covar_est.add(torch.eye(n = sym.shape[0], device = device).mul(0.09))

# Initalise
fig = go.Figure()

fig.add_trace(go.Scatter(x = torch.arange(0, temp_norm.shape[0]), y = temp_norm, # reversing the meaning of x
                        mode = 'lines',
                        name = 'true temperature',
                        line_color = "#05205B"))

fig.add_trace(go.Scatter(x = torch.arange(N_TRAIN.item(), temp_norm.shape[0]), y = x_test_mean_est.cpu(), # reversing the meaning of x
                        mode = 'lines',
                        name = 'mean reconstruction',
                        line_color = 'rgba(0,97,255, 0.8)'))


for i in range(20):
    # Sample via Cholesky
    L = torch.linalg.cholesky(x_test_covar_est_psd.cpu())
    Z = torch.randn(size = [L.shape[-1]], dtype = torch.float32).unsqueeze(0)
    sample = x_test_mean_est.cpu() + torch.matmul(Z, L)  
    fig.add_trace(go.Scatter(x = np.arange(N_TRAIN.item(), co2_norm.shape[0]), y = sample.squeeze(), 
                        mode = 'lines',
                        name = 'Sample',
                        showlegend = False,
                        line = dict(
                                color = 'rgba(0,97,255, 0.4)',
                                width = 0.5
                            )
                        ))

fig.update_layout(template = "simple_white")
fig.update_layout(font_family = "Lato", font_size = 15)
fig.update_layout(legend = dict(x = 1.0, y = 0.9, bgcolor = "rgba(0,0,0,0)"))

fig.update_layout(width = 1000, height = 420)
fig.update_layout(yaxis_range = [-6, 6])

fig.show()

# CO2 -> temp

In [ ]:
k = 7
N_TRAIN = torch.tensor([200]).to(device)

##############
### sigCCM ###
##############

sig_filter = torch.ones(size = (k, )).to(device)
sig_shift = torch.tensor(sig_filter.shape[0] - 1).to(device)
# -1 is towards overlap

NOISE_SCALE = torch.tensor([0.05], device = device)
RBF_SCALE = torch.tensor([2.7], device = device)

y_embeddings, x_gt = embed(
    filter = sig_filter, 
    y = temp_norm, 
    x = co2_norm, 
    max_pos_offset = sig_shift, 
    device = device)

rho, nlml, x_test_mean_est, x_test_covar_est = GP_ccm_sig_predict(
    y_embeddings_train = y_embeddings[0 : N_TRAIN].unsqueeze(-1), # [N, E, 1]
    y_embeddings_test = y_embeddings[N_TRAIN : ].unsqueeze(-1), 
    x_train = x_gt[0 : N_TRAIN], # [N]
    x_test = x_gt[N_TRAIN : ], 
    noise = NOISE_SCALE, 
    rbf_sigma = RBF_SCALE, 
    device = device)

print(rho)

In [ ]:
sym = torch.triu(x_test_covar_est, diagonal = 1).add(torch.triu(x_test_covar_est, diagonal = 1).mT)
# x_test_covar_est_psd = sym.add(torch.eye(n = sym.shape[0], device = device).mul(torch.clip(x_test_covar_est.diag(), min = 0.00001)))
x_test_covar_est_psd = sym.add(torch.eye(n = sym.shape[0], device = device).mul(torch.clip(x_test_covar_est.diag(), min = 0.000).add(0.08)))
# x_test_covar_est_psd_alternative = x_test_covar_est.add(torch.eye(n = sym.shape[0], device = device).mul(0.09))

# Initalise
fig = go.Figure()

fig.add_trace(go.Scatter(x = torch.arange(0, temp_norm.shape[0] -7), y = co2_norm, # reversing the meaning of x
                        mode = 'lines',
                        name = 'true temperature',
                        line_color = "black"))

fig.add_trace(go.Scatter(x = torch.arange(N_TRAIN.item(), temp_norm.shape[0]), y = x_test_mean_est.cpu(), # reversing the meaning of x
                        mode = 'lines',
                        name = 'mean reconstruction',
                        line_color = 'rgba(0,97,255, 0.7)'))


for i in range(20):
    # Sample via Cholesky
    L = torch.linalg.cholesky(x_test_covar_est_psd.cpu())
    Z = torch.randn(size = [L.shape[-1]], dtype = torch.float32).unsqueeze(0)
    sample = x_test_mean_est.cpu() + torch.matmul(Z, L)  
    fig.add_trace(go.Scatter(x = np.arange(N_TRAIN.item(), co2_norm.shape[0]), y = sample.squeeze(), 
                        mode = 'lines',
                        name = 'Sample',
                        showlegend = False,
                        line = dict(
                                color = 'rgba(0,97,255, 0.3)',
                                width = 0.5
                            )
                        ))

fig.update_layout(template = "simple_white")
fig.update_layout(font_family = "Lato")
fig.update_layout(legend = dict(x = 0.01, y = 0.9, bgcolor = "rgba(0,0,0,0)"))

fig.update_layout(width = 1000, height = 500)
fig.update_layout(yaxis_range = [-3.2, 3.0])

fig.show()